# SHAP Values From Scratch (regression)
**Dr. Dave Wanik - OPIM 5512: Data Science Using Python - University of Connecticut**

---------------------------------

We've now cracked open the black box two ways. **Permutation importance** told us *which* features the model leans on. **Partial dependence** told us *how* a feature bends the prediction, on average. Both are **global** stories — one summary for the whole dataset.

**SHAP** answers a different, more personal question:

> *For **this one** prediction, how much did **each** feature push it up or down — away from what we'd normally expect?*

The magic is that SHAP doesn't just hand-wave the credit. It splits it **fairly**, using an idea from game theory that won a Nobel Prize. And here's the part most courses skip: **we are going to compute it BY HAND first** — every coalition, every marginal contribution — and *then* write it in about 15 lines of Python. No black-box library required. Once you've done it on paper, the `shap` package will never feel like magic again.

## Where SHAP fits with what we already know

| Tool | Question it answers | Scope | Output per feature |
|---|---|---|---|
| **Permutation importance** | *Which* features does the model rely on? | **Global** (whole test set) | one number |
| **Partial dependence (PDP/ICE)** | *How* does the prediction respond to a feature? | **Global** (a curve) | a curve |
| **SHAP** | For **this row**, how much did each feature push the prediction up/down? | **Local** (one prediction)... | one number **per row** |

Here's the kicker: once you have SHAP values for **every** row, you can **average** them to recover a global importance (just like permutation importance) **and** scatter them to recover a PDP-style shape. SHAP is the tool that **unifies** the last two lectures — it just refuses to throw away the per-row detail along the way.

## The big idea: a fair payout among teammates

Forget machine learning for a second. Imagine three coworkers — **A**, **B**, and **C** — team up and earn a **\$14 bonus**. How much of that \$14 does each person deserve?

You can't just split it evenly (\$4.67 each) if A did most of the work. And you can't just look at what each person produces *alone*, because some of the value only appears when A and B work **together** (synergy).

In 1953, **Lloyd Shapley** (Nobel Prize, 2012) gave the one and only "fair" answer. His rule:

> A player's fair share = their **average marginal contribution**, taken over **every possible order** in which the team could have been assembled.

"Marginal contribution" just means: *how much bigger did the payout get the moment this player walked in the door?* Do that for every possible arrival order, average it, and you have the **Shapley value**.

**Now swap the words:** the *players* are our **features**, the *payout* is the model's **prediction**, and the *fair share* is how much each feature contributed to *this* prediction. That's SHAP. (SHAP = **SH**apley **A**dditive ex**P**lanations.)

## Step 1 — the "value of a team" (the value function)

To average marginal contributions we first need to score a **team** (a *coalition*) of features. We write this as $v(S)$ = the model's output when **only the features in set $S$ are "present."**

But what does it mean for a feature to be *absent*? We have to put *something* in its slot. For our hand example we'll use the simplest possible choice: a single **baseline** value for each feature (think "a totally average house"). A feature is "present" = use its real value; "absent" = fall back to the baseline.

So:
- $v(\varnothing)$ = prediction when **every** feature is at baseline → our **starting expectation**.
- $v(\text{all features})$ = prediction with **every** real value → **the actual prediction**.
- Everything in between = some features real, some at baseline.

The whole game is to explain the gap between those two: $\; v(\text{all}) - v(\varnothing)$.

## Step 2 — a tiny model we can do on the whiteboard

Let's use a model so simple we can evaluate it in our heads. Three features $a, b, c$:

$$ f(a, b, c) = 5a + 3b + 2c + \underbrace{4ab}_{\text{synergy!}} $$

- **baseline** = $(0, 0, 0)$  → so an "absent" feature contributes nothing.
- the **house we want to explain** = $(a, b, c) = (1, 1, 1)$.

Why this model? Feature **$c$ is a loner** — it never interacts with anyone. But **$a$ and $b$ have synergy**: that $4ab$ term only fires when *both* are present. Watch how SHAP handles that shared credit — it's the whole point.

Let's compute $v(S)$ for all $2^3 = 8$ possible teams.

In [ ]:
import itertools, math
import numpy as np
import matplotlib.pyplot as plt

# our whiteboard model
def f(a, b, c):
    return 5*a + 3*b + 2*c + 4*a*b

names    = ['a', 'b', 'c']
instance = {'a': 1, 'b': 1, 'c': 1}   # the house we are explaining
baseline = {'a': 0, 'b': 0, 'c': 0}   # "absent" = baseline value

def v(S):
    # Value of coalition S: present features use real values, absent fall back to baseline
    S = set(S)
    vals = {k: (instance[k] if k in S else baseline[k]) for k in names}
    return f(vals['a'], vals['b'], vals['c'])

# print the value of every possible team
print("Value of every coalition v(S):\n")
for k in range(len(names)+1):
    for S in itertools.combinations(names, k):
        label = "{" + ",".join(S) + "}" if S else "{} (empty)"
        print(f"  v({label:<12}) = {v(S):>2}")


## Step 3 — the two formulas (they give the same answer)

There are two equivalent ways to write the Shapley value. **Both are worth seeing** because one is intuitive and the other is what we'll actually code.

**(A) The "average over all arrival orders" form** — the intuitive one:

$$ \phi_i = \frac{1}{(\text{number of orderings})}\sum_{\text{every order}} \Big[\, v(\text{team just before } i \text{ joins} + i) - v(\text{team just before } i \text{ joins}) \,\Big] $$

With 3 features there are $3! = 6$ orders. For each order, watch how much the prediction jumps **the moment feature $i$ walks in**, then average those jumps.

**(B) The "weighted over all coalitions" form** — the efficient one:

$$ \phi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!\,\big(p - |S| - 1\big)!}{p!}\,\Big[\, v(S \cup \{i\}) - v(S) \,\Big] $$

where $p$ is the number of features. That ugly fraction is just the **fraction of orderings** in which exactly the players in $S$ arrive before $i$ — so (B) is really (A) with the duplicate orders collected together. We'll compute it **both ways** and confirm they match.

In [ ]:
# ---- Formula (A): average marginal contribution over all 6 arrival orders ----
print("FORMULA A — averaging over every arrival order\n")

contribs = {k: [] for k in names}
for order in itertools.permutations(names):
    team = set()
    for player in order:
        before = v(team)
        team.add(player)
        after = v(team)
        contribs[player].append(after - before)   # the jump when 'player' joined

# show feature 'a' in detail so students can see the averaging
print("Marginal contribution of feature 'a' in each of the 6 orders:")
for order in itertools.permutations(names):
    team = set()
    jump = None
    for player in order:
        before = v(team); team.add(player); after = v(team)
        if player == 'a':
            jump = after - before
    print(f"   order {order} ->  a contributed {jump}")
print(f"   average for a = {np.mean(contribs['a']):.4f}\n")

phi_A = {k: np.mean(contribs[k]) for k in names}
print("Shapley values (Formula A):", {k: round(phi_A[k], 4) for k in names})


In [ ]:
# ---- Formula (B): weighted sum over all coalitions that EXCLUDE feature i ----
print("FORMULA B — weighted sum over coalitions\n")

p = len(names)
def shapley_weight(s_size, p):
    # fraction of orderings where exactly these |S| players come before i
    return math.factorial(s_size) * math.factorial(p - s_size - 1) / math.factorial(p)

phi_B = {}
for i in names:
    others = [k for k in names if k != i]
    total = 0.0
    for k in range(len(others) + 1):
        for S in itertools.combinations(others, k):
            w = shapley_weight(len(S), p)
            marginal = v(set(S) | {i}) - v(set(S))
            total += w * marginal
    phi_B[i] = total

print("Shapley values (Formula B):", {k: round(phi_B[k], 4) for k in names})
print("\nSame answer as Formula A?  ->",
      all(abs(phi_A[k] - phi_B[k]) < 1e-9 for k in names))


## Step 4 — read the answer (this is the beautiful part)

You should have gotten:

$$ \phi_a = 7, \qquad \phi_b = 5, \qquad \phi_c = 2 $$

Let's make sense of every number:

- **$\phi_c = 2$.** Feature $c$ is the loner ($2c$, no interactions). Its credit is exactly its raw effect, $2 \times 1 = 2$. Clean.
- **The synergy gets split fairly.** The $4ab$ term is worth 4 when *both* show up — but who gets the credit? SHAP gives **half to each**: 2 to $a$, 2 to $b$.
  - $\phi_a = \underbrace{5}_{\text{its own } 5a} + \underbrace{2}_{\text{half the synergy}} = 7$
  - $\phi_b = \underbrace{3}_{\text{its own } 3b} + \underbrace{2}_{\text{half the synergy}} = 5$

- **The books balance.** $\;7 + 5 + 2 = 14$, and $14 = f(1,1,1) - f(0,0,0) =$ prediction $-$ baseline. **Every dollar of the prediction is accounted for.** That property has a name — *local accuracy / efficiency* — and it's why SHAP is trustworthy: nothing is invented, nothing leaks.

Let's draw that "every dollar accounted for" story as a **waterfall** — start at the baseline and let each feature push the prediction to its final value.

In [ ]:
# ---- A from-scratch waterfall plot (no shap library needed) ----
phi = phi_B
base_pred = v(set())                 # baseline prediction
order_plot = ['a', 'b', 'c']         # plot features left to right

fig, ax = plt.subplots(figsize=(8, 4.5))
running = base_pred
ax.axhline(base_pred, color='grey', ls='--', lw=1)
ax.text(-0.4, base_pred, f'baseline\n{base_pred:.0f}', va='center', ha='right', fontsize=10)

for idx, k in enumerate(order_plot):
    contrib = phi[k]
    color = '#2ca02c' if contrib >= 0 else '#d62728'   # green up, red down
    ax.bar(idx, contrib, bottom=running, color=color, edgecolor='black', width=0.6)
    ax.text(idx, running + contrib/2, f'{k}\n+{contrib:.0f}', ha='center', va='center',
            color='white', fontweight='bold')
    running += contrib

ax.axhline(running, color='black', lw=1)
ax.text(len(order_plot)-0.6, running, f'prediction\n{running:.0f}', va='center', ha='left', fontsize=10)
ax.set_xticks(range(len(order_plot))); ax.set_xticklabels(order_plot)
ax.set_title("SHAP waterfall (built from scratch): baseline + each feature's push = prediction")
ax.set_ylabel("prediction value"); ax.set_xlim(-1.2, len(order_plot)+0.2)
plt.tight_layout(); plt.show()

print(f"baseline {base_pred:.0f}  +  ({phi['a']:.0f} + {phi['b']:.0f} + {phi['c']:.0f})"
      f"  =  {base_pred + sum(phi.values()):.0f}   (= the prediction)")


## The four properties SHAP guarantees (and where we just saw them)

These aren't trivia — they're *why* SHAP is the explanation method people trust in court, in medicine, and in credit decisions.

1. **Local accuracy (efficiency).** The feature credits sum **exactly** to (prediction − baseline). *We saw it:* $7+5+2 = 14$.
2. **Symmetry.** Two features that contribute identically to **every** coalition get **equal** credit. *We saw it:* $a$ and $b$ split the synergy 50/50.
3. **Dummy / null player.** A feature that never changes any coalition's value gets **exactly 0**. (Add a feature $d$ with coefficient 0 and its $\phi_d = 0$, guaranteed.)
4. **Additivity.** SHAP values of an *ensemble* (say a random forest) equal the **average** of the SHAP values of its trees. This is what makes SHAP play nicely with the ensemble models we love.

No other attribution method satisfies all four at once. That uniqueness is Shapley's theorem.

## ✏️ YOUR TURN — work one out by hand

Don't peek at the solution. Grab paper. Here is a **new** model:

$$ g(a, b, c) = 4a + 2b + 6c + \underbrace{2bc}_{\text{synergy between } b \text{ and } c} $$

Same setup: **baseline** $(0,0,0)$, **instance** $(1,1,1)$.

**Tasks:**
1. Fill in the value of all 8 coalitions $v(S)$ by hand.
2. Compute $\phi_a, \phi_b, \phi_c$ using **either** formula.
3. **Predict before you compute:** which feature is the "loner" now? Who has to *share* credit? Check that your three numbers add up to $g(1,1,1)$.

Use the cell below to check yourself **after** you've done it on paper.

In [ ]:
# ✏️ YOUR TURN: fill in the model, then reuse the machinery above.
def g(a, b, c):
    return 4*a + 2*b + 6*c + 2*b*c     # the new model

def v2(S):
    S = set(S)
    vals = {k: (instance[k] if k in S else baseline[k]) for k in names}
    return g(vals['a'], vals['b'], vals['c'])

# ---- compute Shapley with Formula B (reusing shapley_weight from above) ----
phi_yourturn = {}
for i in names:
    others = [k for k in names if k != i]
    total = 0.0
    for k in range(len(others) + 1):
        for S in itertools.combinations(others, k):
            total += shapley_weight(len(S), p) * (v2(set(S) | {i}) - v2(set(S)))
    phi_yourturn[i] = total

print("Your coalition values:", {("{"+",".join(S)+"}" if S else "{}"): v2(S)
      for kk in range(4) for S in itertools.combinations(names, kk)})
print("Shapley values:", {k: round(phi_yourturn[k], 4) for k in names})
print("Sum =", round(sum(phi_yourturn.values()), 4), " | g(1,1,1) =", g(1,1,1))


```{admonition} 🔑 Solution (instructor — delete before sharing with students)
:class: dropdown
Coalition values: $v(\varnothing)=0,\ v(a)=4,\ v(b)=2,\ v(c)=6,\ v(ab)=6,\ v(ac)=10,\ v(bc)=10,\ v(abc)=14$.

Shapley values: $\boxed{\phi_a = 4,\quad \phi_b = 3,\quad \phi_c = 7}$ — and $4+3+7 = 14 = g(1,1,1)$. ✔

**Now $a$ is the loner** (coefficient 4, no interaction) so $\phi_a = 4$ exactly. This time $b$ and $c$ share the $2bc$ synergy — 1 each — so $\phi_b = 2+1 = 3$ and $\phi_c = 6+1 = 7$. Same fairness rule, different cast.
```

## From the whiteboard to a REAL model

The hand example cheated in one spot: we used a **single baseline** $(0,0,0)$ for "absent." Real models don't have an obvious single baseline — what *is* an "average" house?

The honest fix is to **marginalize**: when a feature is "absent," don't plug in one number — **average the model's output over a whole background sample** of realistic values for that feature. Formally, for a coalition $S$:

$$ v(S) = \frac{1}{N}\sum_{j=1}^{N} f\big(\,\text{instance values on } S,\ \text{background row } j \text{ on the rest}\,\big) $$

In words: *freeze the features we're "keeping" at this house's real values, and for the "absent" features, borrow values from $N$ real background houses and average the predictions.* Everything else — the coalitions, the weights, the averaging — is **identical** to what we just did by hand.

Let's wire it up on **Boston Housing**, exactly like our permutation-importance and PDP notebooks. To keep the math **exact** (and fast), we'll reuse the same "sneaky" 4-feature subset — that's only $2^4 = 16$ coalitions per row.

In [ ]:
# Same recipe as our other Module 2 notebooks - you've seen this a hundred times now!
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

# grab Boston Housing from Drive (same file as the permutation-importance notebook)
!gdown --id 1a0aNGSFWB-pf5ut1NsjE5ECIsbHHoAwI
df = pd.read_csv('BostonHousing.csv')

Y = df['medv']
# the same 4-feature "sneaky subset" from the permutation-importance lecture
feature_names = ['lstat', 'crim', 'dis', 'rad']
X = df[feature_names]

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, shuffle=True, random_state=42)

# scale on TRAIN only (no leakage!) - keep DataFrames so we can read columns by name
scaler = MinMaxScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_test  = pd.DataFrame(scaler.transform(X_test),      columns=feature_names, index=X_test.index)

RFR = RandomForestRegressor(random_state=42).fit(X_train, y_train)
print("Test R2:", round(r2_score(y_test, RFR.predict(X_test)), 3))


In [ ]:
# ---- SHAP from scratch for ANY sklearn model: exact over all 2^p coalitions ----
def shapley_from_scratch(model, x_row, background, feature_names):
    # Exact marginal (interventional) Shapley values for one row.
    #   x_row      : 1D array-like, the instance to explain (length p)
    #   background : 2D array (N x p) of realistic rows used for 'absent' features
    # Returns (dict {feature: shapley_value}, baseline prediction v(empty)).
    x_row = np.asarray(x_row, dtype=float)
    bg    = np.asarray(background, dtype=float)
    p     = len(feature_names)
    idx   = list(range(p))

    # value function v(S): keep S at x_row, borrow the rest from the background, average
    def v(S):
        # value function v(S): keep S at x_row, borrow the rest from the background, average
        S = set(S)
        Z = bg.copy()                         # start from every background row
        for j in S:                           # overwrite the "present" features
            Z[:, j] = x_row[j]
        return model.predict(Z).mean()        # marginalize the absent ones

    phi = {}
    for i in idx:
        others = [j for j in idx if j != i]
        total = 0.0
        for k in range(len(others) + 1):
            for S in itertools.combinations(others, k):
                w = math.factorial(len(S)) * math.factorial(p - len(S) - 1) / math.factorial(p)
                total += w * (v(set(S) | {i}) - v(set(S)))
        phi[feature_names[i]] = total
    return phi, v(set())          # also return the baseline v(empty)

# explain ONE test house using a background sample of training rows
background = X_train.sample(100, random_state=0).values
row_id = 0
x_explain = X_test.iloc[row_id].values

phi_row, base_pred = shapley_from_scratch(RFR, x_explain, background, feature_names)

pred = RFR.predict(x_explain.reshape(1, -1))[0]
print("baseline  E[f(x)] :", round(base_pred, 3))
for k in feature_names:
    print(f"  SHAP[{k:>5}] = {phi_row[k]:+.3f}")
print("-"*34)
print("baseline + sum(SHAP):", round(base_pred + sum(phi_row.values()), 3))
print("actual prediction   :", round(pred, 3), " <- local accuracy still holds!")


In [ ]:
# the same from-scratch waterfall, now on a REAL prediction
fig, ax = plt.subplots(figsize=(8.5, 4.5))
running = base_pred
ax.axhline(base_pred, color='grey', ls='--', lw=1)
ax.text(-0.4, base_pred, f'baseline\n{base_pred:.1f}', va='center', ha='right', fontsize=10)

# sort features by absolute push for a nicer story
ordered = sorted(feature_names, key=lambda k: -abs(phi_row[k]))
for idx_, k in enumerate(ordered):
    c = phi_row[k]
    color = '#2ca02c' if c >= 0 else '#d62728'
    ax.bar(idx_, c, bottom=running, color=color, edgecolor='black', width=0.6)
    ax.text(idx_, running + c/2, f'{k}\n{c:+.2f}', ha='center', va='center',
            color='white', fontweight='bold', fontsize=9)
    running += c

ax.axhline(running, color='black', lw=1)
ax.text(len(ordered)-0.6, running, f'pred\n{running:.1f}', va='center', ha='left', fontsize=10)
ax.set_xticks(range(len(ordered))); ax.set_xticklabels(ordered)
ax.set_title(f"Why the model predicted {pred:.1f} for test house #{row_id}")
ax.set_ylabel("predicted median value ($1000s)")
plt.tight_layout(); plt.show()


## SHAP recovers permutation importance (global from local)

Now the payoff. Compute SHAP for **many** test houses, take the **mean absolute** SHAP value per feature, and you get a **global importance ranking** — built entirely from local explanations. Let's put it side-by-side with `permutation_importance` from the earlier lecture.

In [ ]:
# SHAP values for a batch of test rows (exact; 4 features = cheap)
N_ROWS = 60
rows = X_test.iloc[:N_ROWS].values
shap_matrix = np.array([
    [shapley_from_scratch(RFR, r, background, feature_names)[0][k] for k in feature_names]
    for r in rows
])

mean_abs_shap = np.abs(shap_matrix).mean(axis=0)

# permutation importance from the previous lecture, for comparison
perm = permutation_importance(RFR, X_test, y_test, n_repeats=30, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
o1 = np.argsort(mean_abs_shap)
axes[0].barh(np.array(feature_names)[o1], mean_abs_shap[o1], color='#1f77b4')
axes[0].set_title("Global importance from SHAP\n(mean |SHAP| over rows)")

o2 = np.argsort(perm.importances_mean)
axes[1].barh(np.array(feature_names)[o2], perm.importances_mean[o2], color='#ff7f0e')
axes[1].set_title("Permutation importance\n(previous lecture)")
plt.tight_layout(); plt.show()

print("Same #1 feature both ways?  ->",
      feature_names[int(np.argmax(mean_abs_shap))] ==
      feature_names[int(np.argmax(perm.importances_mean))])


## SHAP also recovers the PDP shape (dependence plot)

Scatter each row's **SHAP value for a feature** against that feature's **actual value**. The cloud traces out *how the prediction responds to the feature* — the same story a **partial dependence plot** tells, but now every dot is a real house and you can see the spread (the interactions PDP averages away).

In [ ]:
feat = 'lstat'
j = feature_names.index(feat)

plt.figure(figsize=(7, 4.5))
plt.scatter(rows[:, j], shap_matrix[:, j], c=rows[:, j], cmap='viridis', s=35, edgecolor='k', lw=0.3)
plt.axhline(0, color='grey', ls='--', lw=1)
plt.xlabel(f"{feat} (scaled)"); plt.ylabel(f"SHAP value for {feat}")
plt.title(f"SHAP dependence plot for '{feat}'  (compare to its PDP!)")
plt.colorbar(label=feat); plt.tight_layout(); plt.show()


## Sanity check: the `shap` library agrees with our hand math

Everything above used **zero** special libraries — just `itertools` and `numpy`. The famous `shap` package is doing **exactly** this calculation (with clever shortcuts so it scales past 4 features). Let's prove our numbers match, and grab its gorgeous **beeswarm** plot for free.

```{admonition} Why our numbers match `shap` exactly
:class: tip
We computed *interventional* (marginal) Shapley values with a background sample. `shap.TreeExplainer(..., feature_perturbation="interventional", data=background)` uses the **same** value function — so for our 4-feature forest the values line up to the decimal.
```

In [ ]:
# optional: install + compare. Wrapped so the notebook still runs if offline.
try:
    import shap
except ImportError:
    !pip -q install shap
    import shap

explainer = shap.TreeExplainer(RFR, data=background, feature_perturbation="interventional")
sv = explainer.shap_values(X_test.iloc[:N_ROWS])

print("Row 0 - ours vs shap:")
for k in feature_names:
    ours = phi_row[k]
    theirs = sv[0][feature_names.index(k)]
    print(f"  {k:>5}:  ours {ours:+.3f}   shap {theirs:+.3f}   match={abs(ours-theirs)<1e-2}")

shap.summary_plot(sv, X_test.iloc[:N_ROWS], feature_names=feature_names, show=True)


## Wrap-up

```{admonition} Key takeaways
:class: tip
- **SHAP explains ONE prediction**: how much each feature pushed it above/below the **baseline** $E[f(x)]$.
- It comes from **game theory** — a feature's value is its **average marginal contribution over every coalition/order**. That's the *only* "fair" split (Shapley's theorem).
- Two equivalent formulas: **average over orderings** (intuitive) and **weighted over coalitions** (what we code). We verified they match.
- **Local accuracy**: baseline + Σ SHAP = the prediction, **exactly**. Plus symmetry, dummy, and additivity.
- For real models we **marginalize absent features over a background sample** — same machinery, just averaged.
- **SHAP unifies the chapter**: mean$|$SHAP$|$ recovers **permutation importance**; a SHAP **dependence plot** recovers the **PDP** — without throwing away the per-row detail.
- The `shap` library is just this, made fast — and our from-scratch numbers match it.
```

**Up next: LIME** — a different route to local explanations (fit a tiny interpretable model *around* one prediction). Once you've built SHAP by hand, LIME's trick will click immediately.

---
*Dr. Dave Wanik — OPIM 5512, University of Connecticut.*